# Figure 3

In [ ]:
options(warn=-1)

In [ ]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 
                         
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ComplexHeatmap), 
            library(patchwork), 
            library(cowplot), 
            library(ggrepel), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv="p.3.8.12-FD20250213HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=4)) # From project global source()

In [37]:
# Source scripts 
source("bin/seurat_qc.R")
source("script/figure/bin/dp_feature.R")
source("script/figure/bin/line_plot.R")
source("script/figure/bin/dea_stats.R")

# Import Seurat object

In [ ]:
so <- readRDS("data/object/subsets/annotation/myeloid.rds")

# UMAP

In [ ]:
dplot_1 <- dplot(so, reduction="umap", group_by="leiden_cell_type_main", alpha=1, pt_size=0.5, size_select=4) +
    ggtitle("Cell type leiden") +
    scale_color_manual(values=color$cell_type_leiden_subset_r, na.value="grey50", labels=base::str2expression(levels(so$leiden_cell_type_main))) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    theme(
    
        legend.text.align=0
    
    ) 

In [ ]:
pdf("result/figures/figure_3/umap_cell_type_leiden_subset_myeloid.pdf", width=10, height=3)

dplot_1

dev.off()

# Marker gene expression 

In [ ]:
# Normalize 
so <- NormalizeData(so, assay="RNA", normalization.method="LogNormalize", scale.factor=1e6)

In [ ]:
# Genes and categories
genes <- read.csv("script/figure/figure_3/data/figure_3_dotplot_celltype_markers.csv")

In [ ]:
dp_1 <- dp_feature(so, genes$gene, group_by="leiden_cell_type_main", group_by_order=rev(levels(so$leiden_cell_type_main)), split=genes$group, split_order=unique(genes$group), range_max=3, scale=TRUE) + 
    scale_y_discrete(labels=base::str2expression(rev(levels(so$leiden_cell_type_main))))

In [ ]:
pdf("result/figures/figure_3/dp_feature_leiden_cell_type_subset.pdf", width=nrow(genes)*0.1+2.5, height=1.5+length(unique(genes$group))*0.1)

dp_1

dev.off()

# Host Donor cell enrichment of skin macrophage 

In [ ]:
so_i <- readRDS("data/object/pp_subset.rds")

In [76]:
data <- so_i@meta.data %>% 
    
    # Filter for skin immune cells 
    dplyr::filter(domain_leiden_subset %in% c("myeloid", "lymphocyte"), hto_demux_class=="Skin") %>% 

    # Select data 
    dplyr::select(patient_id, time_point, cell_type_leiden_subset, genotype_class) %>% droplevels() %>%
    
    # Compute total count immune cells 
    dplyr::group_by(patient_id, time_point) %>% 
    dplyr::mutate(n_cells_total=n()) %>% 

    # Compute total cell count per genotype 
    dplyr::group_by(genotype_class, cell_type_leiden_subset, .add=TRUE) %>% 
    dplyr::mutate(n_cells_gc=n()) %>% 
    
    # Fill in missing genotype class counts 
    dplyr::group_by(n_cells_total, .add=TRUE) %>%
    group_by(across(setdiff(group_vars(.), 'genotype_class'))) %>% 
    tidyr::complete(., genotype_class, fill=list(n_cells_gc=0)) %>%

    # Compute normalized counts 
    dplyr::mutate(norm_cells_gc=10^3*n_cells_gc/n_cells_total) %>%
    
    # Filter by cell type 
    dplyr::filter(cell_type_leiden_subset %in% grep("M*Phi", names(color$cell_type_leiden_subset_r), value=TRUE)) %>% 

    # Ungroup and cleanup 
    dplyr::ungroup() %>% dplyr::distinct()

In [77]:
data <- dplyr::group_by(data, time_point, cell_type_leiden_subset, genotype_class) %>% 
    dplyr::mutate(mean=mean(norm_cells_gc), sd=sd(norm_cells_gc), n=length(unique(patient_id)), sem=sd/sqrt(n), ymin=ifelse(mean-sem<0, 0, mean-sem), ymax=mean+sem) %>%
    dplyr::select(cell_type_leiden_subset, time_point, genotype_class, mean, ymin, ymax) %>% dplyr::distinct()

In [133]:
lp <- lp_time_point(data, title="", ylab="10^3*' x cells / total CD45+ cells'", log_breaks=TRUE) + facet_grid(~cell_type_leiden_subset, labeller=label_parsed)

In [132]:
pdf("result/figures/figure_3/lp_macrophage_skin_genotype_class.pdf", width=3, height=2)

lp

dev.off()

png 
  2

# Enricher 

## Skin macrophage 

In [ ]:
source("plotting_global.R")

enricher <- read.csv("script/figure/figure_3/data/figure_3_macrophage_skin_enricher.csv") %>% 
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=expression_convert(Msig.DB.Hallmark.2020.pathway)) %>%
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=factor(Msig.DB.Hallmark.2020.pathway, levels=unique(Msig.DB.Hallmark.2020.pathway))) %>%
    reshape2::melt(., id.vars=c("Msig.DB.Hallmark.2020.pathway"), variable="cell_type", value.name=c("count"))  %>%
    dplyr::rename(pathway=Msig.DB.Hallmark.2020.pathway) %>%
    dplyr::mutate(cell_type=ifelse(cell_type=="MAC", grep("M*Phi$", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>%
    dplyr::mutate(cell_type=ifelse(cell_type=="MAC.act", grep("Act", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>% 
    dplyr::mutate(cell_type=ifelse(cell_type=="MAC.inflm", grep("Inf", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>%  
    dplyr::mutate(cell_type=factor(cell_type, levels=rev(names(color$cell_type_leiden_subset_r)))) %>% droplevels() %>% 
    tidyr::complete(., pathway, cell_type, fill = list(count=0), explicit = TRUE)

In [ ]:
labels <- names(color$cell_type_leiden_subset_r)
labels <- labels[labels %in% enricher$cell_type]

bp_1 <- ggplot(enricher, aes(x=pathway, y=log10(count+1), fill=cell_type, group=cell_type)) + 
    geom_bar(stat="identity", position="dodge", color="black", size=0.2) + 
    xlab("") + ylab("log10(count+1)") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=parse(text=labels)) + 
    coord_flip() + facet_grid(pathway~., scales="free", space="free") +
    scale_x_discrete(labels=scales::parse_format()) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL, reverse=TRUE)) + 
    theme(
        strip.text = element_blank(), 
        axis.text.y=element_text(vjust=0.5, hjust=1)
    
    ) 

In [ ]:
pdf("result/figures/figure_3/bp_macrophage_skin_enricher.pdf", width=3, height=1.5+0.05*length(unique(enricher$pathway))*length(unique(enricher$cell_type)))

bp_1

dev.off()

## Skin dendritic cells 

In [ ]:
enricher <- read.csv("script/figure/figure_3/data/figure_3_dc_skin_enricher.csv") %>% 
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=expression_convert(Msig.DB.Hallmark.2020.pathway)) %>%
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=factor(Msig.DB.Hallmark.2020.pathway, levels=unique(Msig.DB.Hallmark.2020.pathway))) %>%
    reshape2::melt(., id.vars=c("Msig.DB.Hallmark.2020.pathway"), variable="cell_type", value.name=c("count"))  %>%
    dplyr::rename(pathway=Msig.DB.Hallmark.2020.pathway) %>%
    dplyr::mutate(cell_type=factor(cell_type, levels=rev(names(color$cell_type_leiden_subset_r)))) %>% droplevels() %>%
    tidyr::complete(., pathway, cell_type, fill = list(count=0), explicit = TRUE)

In [ ]:
labels <- names(color$cell_type_leiden_subset_r)
labels <- labels[labels %in% enricher$cell_type]

bp_1 <- ggplot(enricher, aes(x=pathway, y=log10(count+1), fill=cell_type, group=cell_type)) + 
    geom_bar(stat="identity", position="dodge", color="black", size=0.2) + 
    xlab("") + ylab("log10(count+1)") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=parse(text=labels)) + 
    coord_flip() + facet_grid(pathway~., scales="free", space="free") + 
    scale_x_discrete(labels=scales::parse_format()) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL, reverse=TRUE)) + 
    theme(
        strip.text=element_blank(), 
        axis.text.y=element_text(vjust=0.5, hjust=1)
    
    ) 

In [ ]:
pdf("result/figures/figure_3/bp_dc_skin_enricher.pdf", width=3, height=1.5+0.05*length(unique(enricher$pathway))*length(unique(enricher$cell_type)))

bp_1

dev.off()

## Blood monocyte cells 

In [ ]:
enricher <- read.csv("script/figure/figure_3/data/figure_3_monocyte_blood_enricher.csv") %>% 
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=expression_convert(Msig.DB.Hallmark.2020.pathway)) %>%
    dplyr::mutate(Msig.DB.Hallmark.2020.pathway=factor(Msig.DB.Hallmark.2020.pathway, levels=unique(Msig.DB.Hallmark.2020.pathway))) %>%
    reshape2::melt(., id.vars=c("Msig.DB.Hallmark.2020.pathway"), variable="cell_type", value.name=c("count"))  %>%
    dplyr::rename(pathway=Msig.DB.Hallmark.2020.pathway) %>%
    dplyr::mutate(cell_type=ifelse(cell_type=="Monoc", grep("Cl", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>%
    dplyr::mutate(cell_type=ifelse(cell_type=="Monoint", grep("Int", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>% 
    dplyr::mutate(cell_type=ifelse(cell_type=="Mononc", grep("nonCl", names(color$cell_type_leiden_subset_r), value=TRUE), as.character(cell_type))) %>%
    dplyr::mutate(cell_type=factor(cell_type, levels=rev(names(color$cell_type_leiden_subset_r)))) %>% droplevels() %>%
    tidyr::complete(., pathway, cell_type, fill = list(count=0), explicit = TRUE)

In [ ]:
labels <- names(color$cell_type_leiden_subset_r)
labels <- labels[labels %in% enricher$cell_type]

bp_1 <- ggplot(enricher, aes(x=pathway, y=log10(count+1), fill=cell_type, group=cell_type)) + 
    geom_bar(stat="identity", position="dodge", color="black", size=0.2) + 
    xlab("") + ylab("log10(count+1)") + 
    scale_fill_manual(values=color$cell_type_leiden_subset_r, labels=parse(text=labels)) + 
    coord_flip() + facet_grid(pathway~., scales="free", space="free") + 
    scale_x_discrete(labels=scales::parse_format()) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL, reverse=TRUE)) + 
    theme(
        strip.text = element_blank(), 
        axis.text.y=element_text(vjust=0.5, hjust=1)
    
    ) 

In [ ]:
pdf("result/figures/figure_3/bp_monocyte_blood_enricher.pdf", width=3, height=1.5+0.05*length(unique(enricher$pathway))*length(unique(enricher$cell_type)))

bp_1

dev.off()

# DEA Baseline vs GVHD for macrophage and selected genes

In [134]:
genes <- read.csv("script/figure/figure_3/data/figure_3_macrophage_skin_heatmap_deg_baseline_vs_gvhd.csv")

In [135]:
grouping_var <- "time_point"
random_effect_var <- "patient_id"
cell_type <- "leiden_cell_type_main"
cell_min <- 10

In [136]:
# Subset Seurat object
so_i <- subset(so, hto_demux_class=="Skin")

In [137]:
# Get Counts 
cnt <- GetAssayData(so_i, assay="RNA", slot="counts")

In [138]:
# Prepare count data for split 
cnt <- t(as.matrix(cnt))
cnt <- as.data.frame(cnt)

In [139]:
# Make pseudobulks by suming single cell counts
cnt <- split(cnt, f=paste0(so_i[[grouping_var, drop=TRUE]], "|", so_i[[random_effect_var, drop=TRUE]], "|", so_i[[cell_type, drop=TRUE]]))

In [140]:
# Filter by min cell count 
cnt <- cnt[sapply(cnt, function(x) nrow(x)>=cell_min)]

In [141]:
cnt <- lapply(names(cnt), function(i) {x <- data.frame(counts=colSums(cnt[[i]])); colnames(x) <- i; return(x)})
cnt <- do.call(cbind, cnt)

In [142]:
# Subset by cell type 
cnt <- cnt[, grepl("M*Phi", colnames(cnt))]

In [143]:
# Subset by time point
cnt <- cnt[, grepl("Baseline|GVHD", colnames(cnt))]

In [144]:
# Normalize 
mat <- 10^6*sweep(cnt, 2, colSums(cnt),`/`)
mat <- log(mat+1)

In [145]:
# z-score cpm
mat <- t(scale(t(mat), center=TRUE, scale=TRUE))

In [146]:
# Subset by genes 
mat <- mat[genes$gene, ]

In [147]:
# Column order by time points
mat <- mat[, order(factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point)))]

In [148]:
# Column split
column_split <- sapply(strsplit(colnames(mat), "\\|"), `[[`, 3) 
column_split <- factor(column_split, levels=names(color$cell_type_leiden_subset_r))
column_split <- droplevels(column_split)

In [149]:
# Color function - DEG count
color_ramp_mat <- c(rev(brewer.pal(11,"RdBu"))[1], "#ffffff", rev(brewer.pal(11,"RdBu"))[11])
breaks_mat <- seq(-round(quantile(abs(mat), 0.99)), round(quantile(abs(mat), 0.99)), length.out=length(color_ramp_mat))
color_function_mat <- circlize::colorRamp2(breaks_mat, color_ramp_mat) 

In [150]:
hm_list <- lapply(seq_along(levels(column_split)), function(idx) {
    
    # Subset data
    mat <- mat[, column_split==levels(column_split)[idx]]
    
    # Column split 
    column_split_idx <- factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point))
    
    # Get row annotation 
    top_annotation <- ComplexHeatmap::HeatmapAnnotation(

        time_point=column_split_idx %>% droplevels(), 


            annotation_legend_param=list(

                time_point=list(title="Time point", title_gp=gpar(fontsize=8, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), grid_height=unit(2, "mm"), legend_width=unit(2, "mm"), legend_height=unit(2, "mm"), border="asis")
                
            ), 

            col=list(

                time_point=color$time_point
            ), 

            simple_anno_size=unit(2.5, "mm"), 
            show_annotation_name=FALSE, 
            show_legend=TRUE

    )
    
    hm <- ComplexHeatmap::Heatmap(
    
            matrix=mat,

            col=color_function_mat, 
            na_col="white", 

            width=unit(1.25*ncol(mat), "mm"), 
            height=unit(2*nrow(mat), "mm"), 

            row_title=NULL, 
            row_title_gp=gpar(fontsize=8, fontface="bold"), 

            column_title=parse(text=levels(column_split)[idx]), 
            column_title_gp=gpar(fontsize=8, fontface="bold"), 

            row_names_gp=gpar(fontsize=6, fontface="italic"), 
            column_names_gp=gpar(fontsize=8, fontface="plain"), 

            cluster_rows=FALSE, 
            show_row_dend=FALSE,
            row_split=genes$regulation, 
            row_gap=unit(2, "mm"), 
            show_row_names=TRUE, 
            row_names_side="left", 

            cluster_columns=TRUE, 
            cluster_column_slices=FALSE, 
            show_column_dend=FALSE, 
            column_split=column_split_idx, 
            column_gap=unit(1, "mm"), 
            show_column_names=FALSE, 

            top_annotation=top_annotation, 
            left_annotation=NULL, 

            rect_gp=gpar(col=NA, lwd=0, alpha=1), 

            heatmap_legend_param=list(title="z-score", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), legend_width=unit(2, "mm"), grid_height=unit(10, "mm"), legend_height=unit(10, "mm"), at=breaks_mat), 
            show_heatmap_legend=ifelse(idx==1, TRUE, FALSE),

            border=TRUE, 
            border_gp=gpar(col="black", lwd=unit(0.2, "mm")), 

            use_raster=FALSE, raster_by_magick=FALSE

    )

}
                 )

hm <- Reduce(`+`, hm_list)

In [151]:
pdf("result/figures/figure_3/hm_macrophage_skin_heatmap_deg_baseline_vs_gvhd.pdf", width=4, height=6)

draw(hm, column_title_gp=gpar(fontsize=8, fontface="bold"), merge_legends=TRUE) 

dev.off()

png 
  2

# DEA Baseline vs GVHD for DC skin and selected genes

In [152]:
genes <- read.csv("script/figure/figure_3/data/figure_3_dc_skin_heatmap_deg_baseline_vs_gvhd.csv")

In [153]:
grouping_var <- "time_point"
random_effect_var <- "patient_id"
cell_type <- "leiden_cell_type_main"
cell_min <- 10

In [154]:
# Subset Seurat object
so_i <- subset(so, hto_demux_class=="Skin")

In [155]:
# Get Counts 
cnt <- GetAssayData(so_i, assay="RNA", slot="counts")

In [156]:
# Prepare count data for split 
cnt <- t(as.matrix(cnt))
cnt <- as.data.frame(cnt)

In [157]:
# Make pseudobulks by suming single cell counts
cnt <- split(cnt, f=paste0(so_i[[grouping_var, drop=TRUE]], "|", so_i[[random_effect_var, drop=TRUE]], "|", so_i[[cell_type, drop=TRUE]]))

In [158]:
# Filter by min cell count 
cnt <- cnt[sapply(cnt, function(x) nrow(x)>=cell_min)]

In [159]:
cnt <- lapply(names(cnt), function(i) {x <- data.frame(counts=colSums(cnt[[i]])); colnames(x) <- i; return(x)})
cnt <- do.call(cbind, cnt)

In [160]:
# Subset by cell type 
cnt <- cnt[, grepl("cDC1|cDC2|LC$", colnames(cnt))]

In [161]:
# Subset by time point
cnt <- cnt[, grepl("Baseline|GVHD", colnames(cnt))]

In [162]:
# Normalize 
mat <- 10^6*sweep(cnt, 2, colSums(cnt),`/`)
mat <- log(mat+1)

In [163]:
# z-score cpm
mat <- t(scale(t(mat), center=TRUE, scale=TRUE))

In [164]:
# Subset by genes 
mat <- mat[genes$gene, ]

In [165]:
# Column order by time points
mat <- mat[, order(factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point)))]

In [166]:
# Column split
column_split <- sapply(strsplit(colnames(mat), "\\|"), `[[`, 3) 
column_split <- factor(column_split, levels=names(color$cell_type_leiden_subset_r))
column_split <- droplevels(column_split)

In [167]:
# Color function - DEG count
color_ramp_mat <- c(rev(brewer.pal(11,"RdBu"))[1], "#ffffff", rev(brewer.pal(11,"RdBu"))[11])
breaks_mat <- seq(-round(quantile(abs(mat), 0.99)), round(quantile(abs(mat), 0.99)), length.out=length(color_ramp_mat))
color_function_mat <- circlize::colorRamp2(breaks_mat, color_ramp_mat) 

In [168]:
hm_list <- lapply(seq_along(levels(column_split)), function(idx) {
    
    # Subset data
    mat <- mat[, column_split==levels(column_split)[idx]]
    
    # Column split 
    column_split_idx <- factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point))
    
    # Get row annotation 
    top_annotation <- ComplexHeatmap::HeatmapAnnotation(

        time_point=column_split_idx %>% droplevels(), 


            annotation_legend_param=list(

                time_point=list(title="Time point", title_gp=gpar(fontsize=8, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), grid_height=unit(2, "mm"), legend_width=unit(2, "mm"), legend_height=unit(2, "mm"), border="asis")
                
            ), 

            col=list(

                time_point=color$time_point
            ), 

            simple_anno_size=unit(2.5, "mm"), 
            show_annotation_name=FALSE, 
            show_legend=TRUE

    )
    
    hm <- ComplexHeatmap::Heatmap(
    
            matrix=mat,

            col=color_function_mat, 
            na_col="white", 

            width=unit(1.25*ncol(mat), "mm"), 
            height=unit(2*nrow(mat), "mm"), 

            row_title=NULL, 
            row_title_gp=gpar(fontsize=8, fontface="bold"), 

            column_title=parse(text=levels(column_split)[idx]), 
            column_title_gp=gpar(fontsize=8, fontface="bold"), 

            row_names_gp=gpar(fontsize=6, fontface="italic"), 
            column_names_gp=gpar(fontsize=8, fontface="plain"), 

            cluster_rows=FALSE, 
            show_row_dend=FALSE,
            row_split=genes$regulation, 
            row_gap=unit(2, "mm"), 
            show_row_names=TRUE, 
            row_names_side="left", 

            cluster_columns=TRUE, 
            cluster_column_slices=FALSE, 
            show_column_dend=FALSE, 
            column_split=column_split_idx, 
            column_gap=unit(1, "mm"), 
            show_column_names=FALSE, 

            top_annotation=top_annotation, 
            left_annotation=NULL, 

            rect_gp=gpar(col=NA, lwd=0, alpha=1), 

            heatmap_legend_param=list(title="z-score", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), legend_width=unit(2, "mm"), grid_height=unit(10, "mm"), legend_height=unit(10, "mm"), at=breaks_mat), 
            show_heatmap_legend=ifelse(idx==1, TRUE, FALSE),

            border=TRUE, 
            border_gp=gpar(col="black", lwd=unit(0.2, "mm")), 

            use_raster=FALSE, raster_by_magick=FALSE

    )

}
                 )

hm <- Reduce(`+`, hm_list)

In [169]:
pdf("result/figures/figure_3/hm_dc_skin_heatmap_deg_baseline_vs_gvhd.pdf", width=4, height=6)

draw(hm, column_title_gp=gpar(fontsize=8, fontface="bold"), merge_legends=TRUE) 

dev.off()

png 
  2

# DEA overview myeloid cells

## Skin 

In [ ]:
results <- readRDS("result/dea/pb/tx_progression/results_skin.rds")
grouping_stat <- readRDS("result/dea/pb/tx_progression/grouping_stat_skin.rds")

In [ ]:
cell_type_order <- levels(so$leiden_cell_type_main)
cell_type_order <- cell_type_order[cell_type_order %in% names(results)]
p_adj_thr <- 0.05

In [ ]:
results <- results[names(results) %in% cell_type_order]
grouping_stat <- grouping_stat[names(results) %in% cell_type_order]

In [ ]:
source("script/figure/bin/dea_stats.R")

hm_1 <- grouping_stat_hm(grouping_stat, stat="cell_mean", column_title="Cells", name="log(#Cells)", row_order=cell_type_order)
hm_2 <- grouping_stat_hm(grouping_stat, stat="random_effect_var_n", column_title="Patient ID", name="#Patient", row_order=cell_type_order)
hm_3 <- results_n_hm(results, p_adj_thr=p_adj_thr, column_title="DEG", name="log(#DEG)", row_order=cell_type_order)

In [ ]:
pdf("result/figures/figure_3/hm_dea_stat_skin_myeloid.pdf", width=3, height=3)

draw(hm_1 + hm_2 + hm_3, heatmap_legend_side="bottom")

dev.off()

## Blood 

In [ ]:
results <- readRDS("result/dea/pb/tx_progression/results_blood.rds")
grouping_stat <- readRDS("result/dea/pb/tx_progression/grouping_stat_blood.rds")

In [ ]:
cell_type_order <- levels(so$leiden_cell_type_main)
cell_type_order <- cell_type_order[cell_type_order %in% names(results)]
p_adj_thr <- 0.05

In [ ]:
results <- results[names(results) %in% cell_type_order]
grouping_stat <- grouping_stat[names(results) %in% cell_type_order]

In [ ]:
source("script/figure/bin/dea_stats.R")

hm_1 <- grouping_stat_hm(grouping_stat, stat="cell_mean", column_title="Cells", name="log(#Cells)", row_order=cell_type_order)
hm_2 <- grouping_stat_hm(grouping_stat, stat="random_effect_var_n", column_title="Patient ID", name="#Patient", row_order=cell_type_order)
hm_3 <- results_n_hm(results, p_adj_thr=p_adj_thr, column_title="DEG", name="log(#DEG)", row_order=cell_type_order)

In [ ]:
pdf("result/figures/figure_3/hm_dea_stat_blood_myeloid.pdf", width=3, height=3)

draw(hm_1 + hm_2 + hm_3, heatmap_legend_side="bottom")

dev.off()

# DEA volcano plot 

In [ ]:
source("script/figure/bin/dea_vp.R")

## Results for pseudobatch myeloid blood (PB)

In [ ]:
results <- readRDS("result/dea/pb/tx_progression/results_blood.rds")
results <- results[levels(so$leiden_cell_type_main)]

In [ ]:
results <- lapply(results, function(result) result[["GVHD-Baseline"]])
results <- results[!sapply(results, is.null)]

In [ ]:
vp_list <- lapply(names(results), function(i) dea_vp(results[[i]], title=i, color_neg=color$time_point["Baseline"], color_pos=color$time_point["GVHD"]))

In [ ]:
save_plots_to_pdf(vp_list, nrow=4, ncol=3, file_name="result/figures/figure_3/vp_dea_pb_myeloid_blood.pdf")

## Results for pseudobatch myeloid skin (PB)

In [ ]:
results <- readRDS("result/dea/pb/tx_progression/results_skin.rds")
results <- results[levels(so$leiden_cell_type_main)]

In [ ]:
results <- lapply(results, function(result) result[["GVHD-Baseline"]])
results <- results[!sapply(results, is.null)]

In [ ]:
vp_list <- lapply(names(results), function(i) dea_vp(results[[i]], title=i, color_neg=color$time_point["Baseline"], color_pos=color$time_point["GVHD"]))

In [ ]:
save_plots_to_pdf(vp_list, nrow=4, ncol=3, file_name="result/figures/figure_3/vp_dea_pb_myeloid_skin.pdf")